# Baselines

Two reference "models" that every real model has to beat:

1. **Constant**: always predicts the mean `cnt` of the training part.
2. **Hour x day type**: predicts the mean `cnt` of the training part for the
   given pair (`hr`, `workingday`).

Both are fitted **inside each fold** of the validation scheme (via
`evaluate`), so the means never include validation data (no leakage).

In [1]:
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.dummy import DummyRegressor

from src.data import load_data
from src.validation import SEED, evaluate

X, y = load_data()
print(X.shape, y.shape)

(17379, 12) (17379,)


## Hour x day type baseline

A tiny sklearn-compatible estimator. It inherits from `BaseEstimator`, so
`clone` (used in `evaluate`) works. Pairs that were not seen in the training
part fall back to the training mean.

In [2]:
class HourDayBaseline(RegressorMixin, BaseEstimator):
    """Mean of the target for each (hr, workingday) pair of the training part."""

    def fit(self, X, y):
        d = X[["hr", "workingday"]].copy()
        d["y"] = y.to_numpy()
        self.table_ = d.groupby(["hr", "workingday"])["y"].mean()
        self.global_mean_ = float(d["y"].mean())
        return self

    def predict(self, X):
        keys = pd.MultiIndex.from_frame(X[["hr", "workingday"]])
        return self.table_.reindex(keys).fillna(self.global_mean_).to_numpy()

## Evaluation in both schemes

In [3]:
baselines = {
    "baseline_const": DummyRegressor(strategy="mean"),
    "baseline_hour_day": HourDayBaseline(),
}

summary = []
for name, model in baselines.items():
    for scheme in ["random", "chrono"]:
        folds, preds = evaluate(model, X, y, scheme, name)
        summary.append({
            "model": name,
            "scheme": scheme,
            "rmse_mean": folds["rmse"].mean(),
            "rmse_std": folds["rmse"].std(),
            "mae_mean": folds["mae"].mean(),
            "r2_mean": folds["r2"].mean(),
        })

pd.DataFrame(summary).round(3)

,model,scheme,rmse_mean,rmse_std,mae_mean,r2_mean
0,baseline_const,random,181.372,3.540,142.415,-0.001
1,baseline_const,chrono,194.900,45.751,148.262,-0.181
2,baseline_hour_day,random,109.942,1.384,72.959,0.632
3,baseline_hour_day,chrono,125.070,36.325,88.837,0.502


## Per-fold results (chronological scheme)

The folds cover different periods, so their errors are not interchangeable.

In [4]:
pd.read_csv("results/baseline_hour_day_chrono_folds.csv")

,fold,n_train,n_test,test_start,test_end,rmse,mae,r2,rmsle
0,1,2899,2896,2011-05-05 17:00:00,2011-09-03 21:00:00,135.199595,103.595052,0.188731,0.818886
1,2,5795,2896,2011-09-03 22:00:00,2012-01-02 22:00:00,70.621388,47.755813,0.715868,0.543833
2,3,8691,2896,2012-01-02 23:00:00,2012-05-02 23:00:00,112.607021,70.560390,0.598245,0.535099
3,4,11587,2896,2012-05-03 00:00:00,2012-08-31 15:00:00,168.349368,127.359993,0.413626,0.633678
4,5,14483,2896,2012-08-31 16:00:00,2012-12-31 23:00:00,138.574778,94.915973,0.594731,0.595336


## Conclusions

_To be written after looking at the numbers._